# Instalación e importación de librerías

In [1]:
!pip install -q faiss-cpu mlflow optuna lightgbm shap plotly
!unzip -q -o /content/archive.zip -d /content/raw_data

import os, time, random, pickle, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import faiss
import lightgbm as lgb
import optuna
import shap
import mlflow
import plotly.express as px
from scipy.sparse import csr_matrix
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.decomposition import TruncatedSVD

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Fase 1 — Data Processing

In [2]:
# Eventos y categorías
events = pd.read_csv('/content/raw_data/events.csv')
events['timestamp'] = pd.to_datetime(events['timestamp'], unit='ms')
events = events.drop_duplicates(subset=['visitorid', 'event', 'itemid', 'timestamp'])

props = pd.concat([
    pd.read_csv('/content/raw_data/item_properties_part1.csv'),
    pd.read_csv('/content/raw_data/item_properties_part2.csv')
], ignore_index=True)

cat_map = (
    props[props['property'] == 'categoryid']
    .sort_values('timestamp')
    .drop_duplicates('itemid', keep='last')[['itemid', 'value']]
    .rename(columns={'value': 'categoryid'})
)
cat_map['categoryid'] = pd.to_numeric(cat_map['categoryid'], errors='coerce').fillna(-1).astype(int)
del props

# Sesiones, pesos y filtrado k-core
events = events.sort_values(['visitorid', 'timestamp']).reset_index(drop=True)
time_diff = events.groupby('visitorid')['timestamp'].diff().dt.total_seconds().fillna(0)
events['session_id'] = events['visitorid'].astype(str) + '_' + (time_diff > 1800).groupby(events['visitorid']).cumsum().astype(str)
events['weight'] = events['event'].map({'view': 1.0, 'addtocart': 3.0, 'transaction': 5.0})

u_counts = events['visitorid'].value_counts()
i_counts = events['itemid'].value_counts()
df_int = events[
    events['visitorid'].isin(u_counts[u_counts >= 3].index) &
    events['itemid'].isin(i_counts[i_counts >= 5].index)
].merge(cat_map, on='itemid', how='left')

df_int['categoryid'] = df_int['categoryid'].fillna(-1).astype(int)
df_int['user_idx'] = df_int['visitorid'].astype('category').cat.codes
df_int['item_idx'] = df_int['itemid'].astype('category').cat.codes

df_int.to_parquet('/content/interactions.parquet', index=False)
print(f"Interacciones: {len(df_int):,} | Usuarios: {df_int['user_idx'].nunique():,} | Items: {df_int['item_idx'].nunique():,}")

Interacciones: 1,260,733 | Usuarios: 197,028 | Items: 79,873


# Fase 2 — Behavioral Feature Engineering

In [3]:
df = pd.read_parquet('/content/interactions.parquet')

# Split temporal (últimos 7 días para evaluación)
cutoff = df['timestamp'].max() - pd.Timedelta(days=7)
train = df[df['timestamp'] < cutoff].copy()
test = df[df['timestamp'] >= cutoff].copy()

train_users = set(train['user_idx'].unique())
train_items = set(train['item_idx'].unique())
test = test[test['user_idx'].isin(train_users) & test['item_idx'].isin(train_items)].copy()

# Features de usuario
u_agg = train.groupby('user_idx').agg(
    u_events=('event', 'count'),
    u_unique_items=('item_idx', 'nunique'),
    u_sessions=('session_id', 'nunique'),
    u_weight_sum=('weight', 'sum'),
    u_last_ts=('timestamp', 'max')
).reset_index()

u_types = pd.crosstab(train['user_idx'], train['event']).reset_index()
for col in ['view', 'addtocart', 'transaction']:
    if col not in u_types.columns:
        u_types[col] = 0
u_types = u_types.rename(columns={'view': 'u_views', 'addtocart': 'u_carts', 'transaction': 'u_purchases'})

u_fav_cat = (
    train.groupby(['user_idx', 'categoryid'])['weight'].sum()
    .reset_index()
    .sort_values(['user_idx', 'weight'], ascending=[True, False])
    .drop_duplicates('user_idx')[['user_idx', 'categoryid']]
    .rename(columns={'categoryid': 'u_fav_cat'})
)

user_feats = u_agg.merge(u_types, on='user_idx').merge(u_fav_cat, on='user_idx', how='left')
user_feats['u_recency_hrs'] = (cutoff - user_feats['u_last_ts']).dt.total_seconds() / 3600.0
user_feats['u_conv_rate'] = user_feats['u_purchases'] / (user_feats['u_views'] + 1.0)
user_feats = user_feats.drop(columns=['u_last_ts'])

# Features de producto
i_agg = train.groupby('item_idx').agg(
    i_events=('event', 'count'),
    i_unique_users=('user_idx', 'nunique'),
    i_weight_sum=('weight', 'sum'),
    i_last_ts=('timestamp', 'max'),
    categoryid=('categoryid', 'first')
).reset_index()

i_types = pd.crosstab(train['item_idx'], train['event']).reset_index()
for col in ['view', 'addtocart', 'transaction']:
    if col not in i_types.columns:
        i_types[col] = 0
i_types = i_types.rename(columns={'view': 'i_views', 'addtocart': 'i_carts', 'transaction': 'i_purchases'})

item_feats = i_agg.merge(i_types, on='item_idx')
item_feats['i_recency_hrs'] = (cutoff - item_feats['i_last_ts']).dt.total_seconds() / 3600.0
item_feats['i_conv_rate'] = item_feats['i_purchases'] / (item_feats['i_views'] + 1.0)
item_feats = item_feats.drop(columns=['i_last_ts'])

# Guardar artefactos
train.to_parquet('/content/train.parquet', index=False)
test.to_parquet('/content/test.parquet', index=False)
user_feats.to_parquet('/content/user_features.parquet', index=False)
item_feats.to_parquet('/content/item_features.parquet', index=False)

print(f"Train: {len(train):,} | Test: {len(test):,} (Usuarios eval: {test['user_idx'].nunique():,})")
print(f"User feats: {user_feats.shape} | Item feats: {item_feats.shape}")

Train: 1,213,477 | Test: 17,553 (Usuarios eval: 5,142)
User feats: (191023, 11) | Item feats: (79193, 10)


# Fase 3 — Baselines

In [5]:
train = pd.read_parquet('/content/train.parquet')
test = pd.read_parquet('/content/test.parquet')

n_users = max(train['user_idx'].max(), test['user_idx'].max()) + 1
n_items = max(train['item_idx'].max(), test['item_idx'].max()) + 1

# Matriz dispersa y ground truth
ui_train = train.groupby(['user_idx', 'item_idx'])['weight'].sum().reset_index()
R = csr_matrix((ui_train['weight'], (ui_train['user_idx'], ui_train['item_idx'])), shape=(n_users, n_items))

ground_truth = test.groupby('user_idx')['item_idx'].apply(set).to_dict()
eval_users = np.array(list(ground_truth.keys()))
purchased_train = train[train['event'] == 'transaction'].groupby('user_idx')['item_idx'].apply(set).to_dict()

# Métricas Top-K
def eval_recommendations(recs_dict, gt_dict, k=10):
    recalls, precisions, ndcgs, maps, hits = [], [], [], [], []
    idcg_base = np.array([1.0 / np.log2(i + 2) for i in range(k)])

    for u, gt in gt_dict.items():
        preds = recs_dict.get(u, [])[:k]
        if not gt:
            continue
        hit_list = [1 if p in gt else 0 for p in preds]
        n_hits = sum(hit_list)

        hits.append(1.0 if n_hits > 0 else 0.0)
        recalls.append(n_hits / len(gt))
        precisions.append(n_hits / k)

        dcg = sum(h * idcg_base[i] for i, h in enumerate(hit_list))
        idcg = sum(idcg_base[:min(len(gt), k)])
        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)

        ap, running_hits = 0.0, 0
        for i, h in enumerate(hit_list):
            if h:
                running_hits += 1
                ap += running_hits / (i + 1)
        maps.append(ap / min(len(gt), k))

    return {
        f'Recall@{k}': round(np.mean(recalls) * 100, 3),
        f'Precision@{k}': round(np.mean(precisions) * 100, 3),
        f'NDCG@{k}': round(np.mean(ndcgs) * 100, 3),
        f'MAP@{k}': round(np.mean(maps) * 100, 3),
        f'HitRate@{k}': round(np.mean(hits) * 100, 3)
    }

def extract_topk(scores_matrix, users, k=10):
    recs = {}
    for idx, u in enumerate(users):
        row = scores_matrix[idx].copy()
        excl = list(purchased_train.get(u, []))
        if excl:
            row[excl] = -np.inf
        top_idx = np.argpartition(row, -k)[-k:]
        recs[u] = top_idx[np.argsort(row[top_idx])[::-1]].tolist()
    return recs

# 1. Popularity
pop_items = ui_train.groupby('item_idx')['weight'].sum().sort_values(ascending=False).index.tolist()
recs_pop = {u: [i for i in pop_items if i not in purchased_train.get(u, set())][:10] for u in eval_users}

# 2. Item-Based CF
R_item_norm = normalize(R, norm='l2', axis=0)
scores_cf = (R[eval_users] @ R_item_norm.T) @ R_item_norm
scores_cf = scores_cf.toarray() if hasattr(scores_cf, 'toarray') else scores_cf
recs_cf = extract_topk(scores_cf, eval_users, k=10)

# 3. Matrix Factorization (SVD)
svd = TruncatedSVD(n_components=64, random_state=SEED)
U_factors = svd.fit_transform(R)
scores_svd = U_factors[eval_users] @ svd.components_
recs_svd = extract_topk(scores_svd, eval_users, k=10)

# Resultados
baseline_results = []
for name, recs in [('Popularity', recs_pop), ('Item-Based CF', recs_cf), ('Matrix Factorization (SVD)', recs_svd)]:
    row = {'Modelo': name}
    row.update(eval_recommendations(recs, ground_truth, k=10))
    baseline_results.append(row)

df_baselines = pd.DataFrame(baseline_results)
df_baselines.to_csv('/content/baseline_metrics.csv', index=False)
print(df_baselines.to_string(index=False))

                    Modelo  Recall@10  Precision@10  NDCG@10  MAP@10  HitRate@10
                Popularity      0.635         0.150    0.455   0.308       1.420
             Item-Based CF     27.252         3.785   24.975  23.064      31.933
Matrix Factorization (SVD)      2.422         0.443    1.834   1.483       3.287


# Fase 4 — Two-Tower Neural Network

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo: {device}")

train = pd.read_parquet('/content/train.parquet')
test = pd.read_parquet('/content/test.parquet')
user_feats = pd.read_parquet('/content/user_features.parquet')
item_feats = pd.read_parquet('/content/item_features.parquet')

n_users = max(train['user_idx'].max(), test['user_idx'].max()) + 1
n_items = max(train['item_idx'].max(), test['item_idx'].max()) + 1

# Features densas y categorías alineadas
u_cols = ['u_events', 'u_unique_items', 'u_sessions', 'u_weight_sum', 'u_views', 'u_carts', 'u_purchases', 'u_recency_hrs', 'u_conv_rate']
i_cols = ['i_events', 'i_unique_users', 'i_weight_sum', 'i_views', 'i_carts', 'i_purchases', 'i_recency_hrs', 'i_conv_rate']

u_dense_df = user_feats.set_index('user_idx').reindex(range(n_users)).fillna(0)
i_dense_df = item_feats.set_index('item_idx').reindex(range(n_items)).fillna(0)

u_scaler = StandardScaler()
i_scaler = StandardScaler()
U_dense = torch.tensor(u_scaler.fit_transform(u_dense_df[u_cols]), dtype=torch.float32)
I_dense = torch.tensor(i_scaler.fit_transform(i_dense_df[i_cols]), dtype=torch.float32)

cat_series = i_dense_df['categoryid'].fillna(-1).astype('category')
I_cats = torch.tensor(cat_series.cat.codes.values, dtype=torch.long)
n_cats = len(cat_series.cat.categories)

# Pares positivos
pairs = train.groupby(['user_idx', 'item_idx'])['weight'].sum().reset_index()

class InteractionDataset(Dataset):
    def __init__(self, users, items):
        self.users = torch.tensor(users, dtype=torch.long)
        self.items = torch.tensor(items, dtype=torch.long)
    def __len__(self):
        return len(self.users)
    def __getitem__(self, idx):
        return self.users[idx], self.items[idx]

loader = DataLoader(InteractionDataset(pairs['user_idx'].values, pairs['item_idx'].values), batch_size=1024, shuffle=True, drop_last=True)

# Arquitectura Two-Tower
class TwoTowerNet(nn.Module):
    def __init__(self, n_users, n_items, n_cats, u_dim, i_dim, emb_dim=64, out_dim=64):
        super().__init__()
        self.u_emb = nn.Embedding(n_users, emb_dim)
        self.u_mlp = nn.Sequential(
            nn.Linear(emb_dim + u_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, out_dim)
        )
        self.i_emb = nn.Embedding(n_items, emb_dim)
        self.c_emb = nn.Embedding(n_cats, 16)
        self.i_mlp = nn.Sequential(
            nn.Linear(emb_dim + 16 + i_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, out_dim)
        )
        self.temp = 0.07

    def encode_user(self, u_idx, u_feat):
        x = torch.cat([self.u_emb(u_idx), u_feat], dim=-1)
        return F.normalize(self.u_mlp(x), p=2, dim=-1)

    def encode_item(self, i_idx, c_idx, i_feat):
        x = torch.cat([self.i_emb(i_idx), self.c_emb(c_idx), i_feat], dim=-1)
        return F.normalize(self.i_mlp(x), p=2, dim=-1)

    def forward(self, u_idx, u_feat, i_idx, c_idx, i_feat):
        u_vec = self.encode_user(u_idx, u_feat)
        i_vec = self.encode_item(i_idx, c_idx, i_feat)
        return torch.matmul(u_vec, i_vec.T) / self.temp

model = TwoTowerNet(n_users, n_items, n_cats, len(u_cols), len(i_cols)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

U_dense_dev = U_dense.to(device)
I_dense_dev = I_dense.to(device)
I_cats_dev = I_cats.to(device)

# Entrenamiento contrastivo
mlflow.set_experiment("retailrocket_two_tower")
with mlflow.start_run(run_name="two_tower_v1"):
    for epoch in range(1, 7):
        model.train()
        total_loss = 0.0
        for u_b, i_b in loader:
            u_b, i_b = u_b.to(device), i_b.to(device)
            labels = torch.arange(len(u_b), device=device)

            optimizer.zero_grad()
            logits = model(u_b, U_dense_dev[u_b], i_b, I_cats_dev[i_b], I_dense_dev[i_b])
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(loader)
        mlflow.log_metric("train_loss", avg_loss, step=epoch)
        print(f"Epoch {epoch}/6 | Loss: {avg_loss:.4f}")

# Guardado de embeddings y artefactos
model.eval()
with torch.no_grad():
    all_u_idx = torch.arange(n_users, device=device)
    all_i_idx = torch.arange(n_items, device=device)
    user_embs = model.encode_user(all_u_idx, U_dense_dev).cpu().numpy()
    item_embs = model.encode_item(all_i_idx, I_cats_dev, I_dense_dev).cpu().numpy()

np.save('/content/user_embeddings.npy', user_embs)
np.save('/content/item_embeddings.npy', item_embs)
torch.save(model.state_dict(), '/content/two_tower_weights.pt')
with open('/content/scalers.pkl', 'wb') as f:
    pickle.dump({
        'u_scaler': u_scaler,
        'i_scaler': i_scaler,
        'u_cols': u_cols,
        'i_cols': i_cols,
        'cat_categories': list(cat_series.cat.categories)
    }, f)

print(f"Embeddings -> Users: {user_embs.shape} | Items: {item_embs.shape}")

Dispositivo: cuda


2026/11/15 05:52:14 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/15 05:52:14 INFO mlflow.store.db.utils: Updating database tables
2026/11/15 05:52:16 INFO mlflow.tracking.fluent: Experiment with name 'retailrocket_two_tower' does not exist. Creating a new experiment.


Epoch 1/6 | Loss: 6.8376
Epoch 2/6 | Loss: 6.5985
Epoch 3/6 | Loss: 6.0995
Epoch 4/6 | Loss: 5.5729
Epoch 5/6 | Loss: 5.1470
Epoch 6/6 | Loss: 4.8142
Embeddings -> Users: (197028, 64) | Items: (79873, 64)


# Fase 5 — Vector Retrieval

In [7]:
train = pd.read_parquet('/content/train.parquet')
test = pd.read_parquet('/content/test.parquet')
user_embs = np.load('/content/user_embeddings.npy').astype('float32')
item_embs = np.load('/content/item_embeddings.npy').astype('float32')

# Normalización L2 y construcción de índice FAISS (Inner Product = Cosine)
faiss.normalize_L2(user_embs)
faiss.normalize_L2(item_embs)

dim = item_embs.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(item_embs)
faiss.write_index(index, '/content/faiss_item.index')

# Búsqueda Top-100 candidatos y medición de latencia
ground_truth = test.groupby('user_idx')['item_idx'].apply(set).to_dict()
eval_users = np.array(list(ground_truth.keys()))
purchased_train = train[train['event'] == 'transaction'].groupby('user_idx')['item_idx'].apply(set).to_dict()

TOP_K_CAND = 100
t0 = time.perf_counter()
scores, indices = index.search(user_embs[eval_users], TOP_K_CAND + 20)
latency_ms = ((time.perf_counter() - t0) / len(eval_users)) * 1000

# Filtrado de compras previas y armado de candidatos
recs_tt_10 = {}
cand_rows = []

for idx, u in enumerate(eval_users):
    excl = purchased_train.get(u, set())
    filtered_items, filtered_scores = [], []

    for item_id, score in zip(indices[idx], scores[idx]):
        if item_id != -1 and item_id not in excl:
            filtered_items.append(int(item_id))
            filtered_scores.append(float(score))
            if len(filtered_items) == TOP_K_CAND:
                break

    recs_tt_10[u] = filtered_items[:10]
    for rank, (item_id, score) in enumerate(zip(filtered_items, filtered_scores)):
        cand_rows.append((int(u), item_id, score, rank))

df_candidates = pd.DataFrame(cand_rows, columns=['user_idx', 'item_idx', 'tt_score', 'faiss_rank'])
df_candidates.to_parquet('/content/eval_candidates.parquet', index=False)

# Evaluación de recuperación (Top-10 y cobertura Recall@100)
tt_metrics = eval_recommendations(recs_tt_10, ground_truth, k=10)
recall_100 = np.mean([
    len(set(df_candidates[df_candidates['user_idx'] == u]['item_idx']) & gt) / len(gt)
    for u, gt in ground_truth.items() if gt
]) * 100

print(f"Índice FAISS: {index.ntotal:,} ítems | Latencia media: {latency_ms:.4f} ms/usuario")
print(f"Recall@100 (Cobertura de candidatos): {recall_100:.3f}%")
print("Métricas Two-Tower + FAISS (Top-10):", tt_metrics)

Índice FAISS: 79,873 ítems | Latencia media: 0.2063 ms/usuario
Recall@100 (Cobertura de candidatos): 9.534%
Métricas Two-Tower + FAISS (Top-10): {'Recall@10': np.float64(2.891), 'Precision@10': np.float64(0.482), 'NDCG@10': np.float64(1.615), 'MAP@10': np.float64(1.036), 'HitRate@10': np.float64(4.162)}


# Fase 6 — Learning to Rank

In [9]:
train = pd.read_parquet('/content/train.parquet')
test = pd.read_parquet('/content/test.parquet')
user_feats = pd.read_parquet('/content/user_features.parquet')
item_feats = pd.read_parquet('/content/item_features.parquet')
eval_candidates = pd.read_parquet('/content/eval_candidates.parquet')
user_embs = np.load('/content/user_embeddings.npy').astype('float32')
index = faiss.read_index('/content/faiss_item.index')

# Afinidad histórica usuario-ítem
hist_ui = train.groupby(['user_idx', 'item_idx'])['weight'].sum().reset_index().rename(columns={'weight': 'hist_ui_weight'})

def build_ltr_features(cand_df):
    df_f = (
        cand_df
        .merge(user_feats, on='user_idx', how='left')
        .merge(item_feats, on='item_idx', how='left')
        .merge(hist_ui, on=['user_idx', 'item_idx'], how='left')
        .fillna(0)
    )
    df_f['same_fav_cat'] = (df_f['u_fav_cat'] == df_f['categoryid']).astype(np.float32)
    df_f['pop_x_sim'] = df_f['tt_score'] * np.log1p(df_f['i_weight_sum'])
    return df_f

# Dataset de entrenamiento LTR (últimos 7 días de train)
ltr_cutoff = train['timestamp'].max() - pd.Timedelta(days=7)
ltr_target = train[train['timestamp'] >= ltr_cutoff].copy()
rel_map = {'view': 1, 'addtocart': 2, 'transaction': 3}
ltr_target['rel'] = ltr_target['event'].map(rel_map)
ltr_gt = ltr_target.groupby(['user_idx', 'item_idx'])['rel'].max().reset_index()

ltr_users = ltr_target['user_idx'].unique()
scores_tr, indices_tr = index.search(user_embs[ltr_users], 50)

tr_rows = []
for idx, u in enumerate(ltr_users):
    for rank, (item_id, score) in enumerate(zip(indices_tr[idx], scores_tr[idx])):
        if item_id != -1:
            tr_rows.append((int(u), int(item_id), float(score), rank))

df_ltr = pd.DataFrame(tr_rows, columns=['user_idx', 'item_idx', 'tt_score', 'faiss_rank'])
df_ltr = df_ltr.merge(ltr_gt, on=['user_idx', 'item_idx'], how='left')
df_ltr['rel'] = df_ltr['rel'].fillna(0).astype(int)

valid_groups = df_ltr.groupby('user_idx')['rel'].sum()
valid_users = valid_groups[valid_groups > 0].index.values
df_ltr = df_ltr[df_ltr['user_idx'].isin(valid_users)].copy()
df_ltr = build_ltr_features(df_ltr).sort_values('user_idx').reset_index(drop=True)

feature_cols = [
    'tt_score', 'faiss_rank', 'same_fav_cat', 'pop_x_sim', 'hist_ui_weight',
    'u_events', 'u_unique_items', 'u_sessions', 'u_weight_sum', 'u_views',
    'u_carts', 'u_purchases', 'u_recency_hrs', 'u_conv_rate',
    'i_events', 'i_unique_users', 'i_weight_sum', 'i_views',
    'i_carts', 'i_purchases', 'i_recency_hrs', 'i_conv_rate'
]

# Split 80/20 por usuario
rng = np.random.default_rng(SEED)
shuffled_u = rng.permutation(valid_users)
split_idx = int(len(shuffled_u) * 0.8)
u_tr_set = set(shuffled_u[:split_idx])

train_mask = df_ltr['user_idx'].isin(u_tr_set)
X_tr, y_tr = df_ltr.loc[train_mask, feature_cols], df_ltr.loc[train_mask, 'rel']
g_tr = df_ltr.loc[train_mask].groupby('user_idx', sort=False).size().values

X_va, y_va = df_ltr.loc[~train_mask, feature_cols], df_ltr.loc[~train_mask, 'rel']
g_va = df_ltr.loc[~train_mask].groupby('user_idx', sort=False).size().values

# Entrenamiento LightGBM Ranker
ranker = lgb.LGBMRanker(
    objective='lambdarank',
    metric='ndcg',
    ndcg_eval_at=[10],
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    subsample_for_bin=50000,
    random_state=SEED,
    n_jobs=-1
)

ranker.fit(
    X_tr, y_tr,
    group=g_tr,
    eval_set=[(X_va, y_va)],
    eval_group=[g_va],
    callbacks=[lgb.early_stopping(30, verbose=False)]
)

# Re-ranking en evaluación
df_eval_ltr = build_ltr_features(eval_candidates)
df_eval_ltr['ltr_score'] = ranker.predict(df_eval_ltr[feature_cols])
df_eval_ltr = df_eval_ltr.sort_values(['user_idx', 'ltr_score'], ascending=[True, False])

recs_ltr = df_eval_ltr.groupby('user_idx')['item_idx'].apply(lambda s: s.tolist()[:10]).to_dict()
ground_truth = test.groupby('user_idx')['item_idx'].apply(set).to_dict()

ltr_metrics = eval_recommendations(recs_ltr, ground_truth, k=10)
print("Métricas Two-Tower + FAISS + LightGBM Ranker (Top-10):", ltr_metrics)

# Guardar artefactos
df_ltr.to_parquet('/content/ltr_train_dataset.parquet', index=False)
df_eval_ltr[['user_idx', 'item_idx', 'ltr_score'] + feature_cols].to_parquet('/content/eval_ltr_scored.parquet', index=False)
ranker.booster_.save_model('/content/lgbm_ranker.txt')
with open('/content/ltr_meta.pkl', 'wb') as f:
    pickle.dump({'feature_cols': feature_cols, 'valid_users': valid_users}, f)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.018158 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2798
[LightGBM] [Info] Number of data points in the train set: 182550, number of used features: 22
Métricas Two-Tower + FAISS + LightGBM Ranker (Top-10): {'Recall@10': np.float64(7.579), 'Precision@10': np.float64(1.198), 'NDCG@10': np.float64(7.253), 'MAP@10': np.float64(6.592), 'HitRate@10': np.float64(9.743)}


# Fase 7 — Evaluation, Optuna & SHAP

In [10]:
test = pd.read_parquet('/content/test.parquet')
df_ltr = pd.read_parquet('/content/ltr_train_dataset.parquet')
df_eval = pd.read_parquet('/content/eval_ltr_scored.parquet')
df_baselines = pd.read_csv('/content/baseline_metrics.csv')

with open('/content/ltr_meta.pkl', 'rb') as f:
    meta = pickle.load(f)
feature_cols = meta['feature_cols']
valid_users = meta['valid_users']

ground_truth = test.groupby('user_idx')['item_idx'].apply(set).to_dict()

# Split validación para Optuna
rng = np.random.default_rng(SEED)
shuffled_u = rng.permutation(valid_users)
u_tr_set = set(shuffled_u[:int(len(shuffled_u) * 0.8)])

train_mask = df_ltr['user_idx'].isin(u_tr_set)
X_tr, y_tr = df_ltr.loc[train_mask, feature_cols], df_ltr.loc[train_mask, 'rel']
g_tr = df_ltr.loc[train_mask].groupby('user_idx', sort=False).size().values

X_va, y_va = df_ltr.loc[~train_mask, feature_cols], df_ltr.loc[~train_mask, 'rel']
g_va = df_ltr.loc[~train_mask].groupby('user_idx', sort=False).size().values

# Optimización de hiperparámetros con Optuna
def objective(trial):
    params = {
        'objective': 'lambdarank',
        'metric': 'ndcg',
        'ndcg_eval_at': [10],
        'n_estimators': 300,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 63),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'random_state': SEED,
        'n_jobs': -1,
        'verbose': -1
    }
    model_opt = lgb.LGBMRanker(**params)
    model_opt.fit(
        X_tr, y_tr,
        group=g_tr,
        eval_set=[(X_va, y_va)],
        eval_group=[g_va],
        callbacks=[lgb.early_stopping(25, verbose=False)]
    )
    return model_opt.best_score_['valid_0']['ndcg@10']

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=15)
print(f"Mejor NDCG@10 Validación (Optuna): {study.best_value:.5f}")

# Entrenamiento modelo final optimizado y registro MLflow
mlflow.set_experiment("retailrocket_ltr_ranker")
with mlflow.start_run(run_name="lgbm_ranker_optuna"):
    best_params = {
        'objective': 'lambdarank',
        'metric': 'ndcg',
        'ndcg_eval_at': [10],
        'n_estimators': 400,
        'random_state': SEED,
        'n_jobs': -1,
        'verbose': -1,
        **study.best_params
    }
    mlflow.log_params(study.best_params)

    tuned_ranker = lgb.LGBMRanker(**best_params)
    tuned_ranker.fit(
        X_tr, y_tr,
        group=g_tr,
        eval_set=[(X_va, y_va)],
        eval_group=[g_va],
        callbacks=[lgb.early_stopping(30, verbose=False)]
    )

    df_eval['tuned_score'] = tuned_ranker.predict(df_eval[feature_cols])

    # Extracción de Top-10 por etapa del pipeline
    recs_tt = df_eval.sort_values(['user_idx', 'faiss_rank']).groupby('user_idx')['item_idx'].apply(lambda s: s.tolist()[:10]).to_dict()
    recs_ltr_base = df_eval.sort_values(['user_idx', 'ltr_score'], ascending=[True, False]).groupby('user_idx')['item_idx'].apply(lambda s: s.tolist()[:10]).to_dict()
    recs_ltr_tuned = df_eval.sort_values(['user_idx', 'tuned_score'], ascending=[True, False]).groupby('user_idx')['item_idx'].apply(lambda s: s.tolist()[:10]).to_dict()

    stage_rows = []
    for name, recs in [
        ('Two-Tower + FAISS', recs_tt),
        ('Two-Tower + LightGBM (Base)', recs_ltr_base),
        ('Two-Tower + LightGBM (Optuna)', recs_ltr_tuned)
    ]:
        metrics = eval_recommendations(recs, ground_truth, k=10)
        stage_rows.append({'Modelo': name, **metrics})

    df_final = pd.concat([df_baselines, pd.DataFrame(stage_rows)], ignore_index=True)
    df_final.to_csv('/content/final_benchmark.csv', index=False)

    final_metrics = stage_rows[-1]
    for k_m, v_m in final_metrics.items():
        if k_m != 'Modelo':
            mlflow.log_metric(k_m.replace('@', '_at_'), v_m)

print("\n=== BENCHMARK COMPLETO END-TO-END (TOP-10) ===")
print(df_final.to_string(index=False))

# Guardar modelo final y predicciones
tuned_ranker.booster_.save_model('/content/lgbm_ranker_tuned.txt')
df_eval.to_parquet('/content/final_eval_scored.parquet', index=False)

# Interpretabilidad con SHAP
sample_X = df_eval[feature_cols].sample(n=min(5000, len(df_eval)), random_state=SEED)
explainer = shap.TreeExplainer(tuned_ranker)
shap_values = explainer.shap_values(sample_X)

shap_imp = pd.DataFrame({
    'Feature': feature_cols,
    'Mean_Abs_SHAP': np.abs(shap_values).mean(axis=0)
}).sort_values('Mean_Abs_SHAP', ascending=True)

fig_shap = px.bar(
    shap_imp.tail(12),
    x='Mean_Abs_SHAP',
    y='Feature',
    orientation='h',
    title='Top 12 Features por Impacto SHAP en Ranking'
)
fig_shap.show()

fig_bench = px.bar(
    df_final.melt(id_vars='Modelo', value_vars=['Recall@10', 'NDCG@10', 'HitRate@10']),
    x='Modelo',
    y='value',
    color='variable',
    barmode='group',
    title='Comparativa de Métricas Top-10 por Arquitectura'
)
fig_bench.show()

2026/11/15 05:57:49 INFO mlflow.tracking.fluent: Experiment with name 'retailrocket_ltr_ranker' does not exist. Creating a new experiment.


Mejor NDCG@10 Validación (Optuna): 0.99167

=== BENCHMARK COMPLETO END-TO-END (TOP-10) ===
                       Modelo  Recall@10  Precision@10  NDCG@10  MAP@10  HitRate@10
                   Popularity      0.635         0.150    0.455   0.308       1.420
                Item-Based CF     27.252         3.785   24.975  23.064      31.933
   Matrix Factorization (SVD)      2.422         0.443    1.834   1.483       3.287
            Two-Tower + FAISS      2.891         0.482    1.615   1.036       4.162
  Two-Tower + LightGBM (Base)      7.579         1.198    7.253   6.592       9.743
Two-Tower + LightGBM (Optuna)      8.116         1.317    7.628   6.873      10.657


# Fase 8 — Inference Pipeline & Portfolio Packaging

In [12]:
# Mapeos de IDs reales <-> índices internos
df_int = pd.read_parquet('/content/interactions.parquet', columns=['visitorid', 'user_idx', 'itemid', 'item_idx'])
train = pd.read_parquet('/content/train.parquet', columns=['user_idx', 'item_idx', 'event', 'weight'])

visitor_to_idx = df_int.drop_duplicates('visitorid').set_index('visitorid')['user_idx'].to_dict()
idx_to_visitor = df_int.drop_duplicates('user_idx').set_index('user_idx')['visitorid'].to_dict()
idx_to_item = df_int.drop_duplicates('item_idx').set_index('item_idx')['itemid'].to_dict()
del df_int

# Tablas de consulta rápida en memoria (con soporte cold-start)
user_embs_tmp = np.load('/content/user_embeddings.npy')
n_users_total = len(user_embs_tmp)
del user_embs_tmp

u_lookup = pd.read_parquet('/content/user_features.parquet').set_index('user_idx').reindex(range(n_users_total)).fillna(0)
i_lookup = pd.read_parquet('/content/item_features.parquet').set_index('item_idx')
hist_lookup = train.groupby(['user_idx', 'item_idx'])['weight'].sum().to_dict()
purchased_lookup = train[train['event'] == 'transaction'].groupby('user_idx')['item_idx'].apply(set).to_dict()

with open('/content/ltr_meta.pkl', 'rb') as f:
    feature_cols = pickle.load(f)['feature_cols']

# Pipeline de inferencia end-to-end
class RetailRocketRecommender:
    def __init__(self, user_embs_path, faiss_path, ranker_path, feat_cols):
        self.user_embs = np.load(user_embs_path).astype('float32')
        faiss.normalize_L2(self.user_embs)
        self.index = faiss.read_index(faiss_path)
        self.ranker = lgb.Booster(model_file=ranker_path)
        self.feat_cols = feat_cols

    def recommend(self, visitor_id, top_k=10, n_candidates=100):
        t0 = time.perf_counter()
        if visitor_id not in visitor_to_idx:
            return pd.DataFrame(), 0.0

        u_idx = visitor_to_idx[visitor_id]
        excl = purchased_lookup.get(u_idx, set())

        # 1. Recuperación vectorial con FAISS
        scores, indices = self.index.search(self.user_embs[u_idx:u_idx+1], n_candidates + 20)
        cands = []
        for item_id, score in zip(indices[0], scores[0]):
            if item_id != -1 and item_id not in excl:
                cands.append((int(item_id), float(score), len(cands)))
                if len(cands) == n_candidates:
                    break

        df_c = pd.DataFrame(cands, columns=['item_idx', 'tt_score', 'faiss_rank'])
        df_c['user_idx'] = u_idx

        # 2. Ensamblaje de features en tiempo real
        u_row = u_lookup.loc[u_idx]
        for col in u_lookup.columns:
            df_c[col] = u_row[col]

        df_c = df_c.merge(i_lookup, on='item_idx', how='left').fillna(0)
        df_c['hist_ui_weight'] = [hist_lookup.get((u_idx, iid), 0.0) for iid in df_c['item_idx']]
        df_c['same_fav_cat'] = (df_c['u_fav_cat'] == df_c['categoryid']).astype(np.float32)
        df_c['pop_x_sim'] = df_c['tt_score'] * np.log1p(df_c['i_weight_sum'])

        # 3. Re-ranking con LightGBM
        df_c['ltr_score'] = self.ranker.predict(df_c[self.feat_cols])
        top_df = df_c.sort_values('ltr_score', ascending=False).head(top_k).copy()
        top_df['itemid'] = top_df['item_idx'].map(idx_to_item)
        top_df['rank'] = range(1, len(top_df) + 1)

        latency_ms = (time.perf_counter() - t0) * 1000
        return top_df[['rank', 'itemid', 'categoryid', 'tt_score', 'ltr_score']], latency_ms

# Prueba de inferencia en vivo con usuario activo
recommender = RetailRocketRecommender(
    '/content/user_embeddings.npy',
    '/content/faiss_item.index',
    '/content/lgbm_ranker_tuned.txt',
    feature_cols
)

sample_visitor = idx_to_visitor[int(train['user_idx'].iloc[0])]
recs_demo, lat_ms = recommender.recommend(sample_visitor, top_k=10)
print(f"Visitor ID: {sample_visitor} | Latencia End-to-End: {lat_ms:.2f} ms")
print(recs_demo.to_string(index=False))

# Empaquetado de artefactos para repositorio GitHub
os.makedirs('/content/portfolio_release/artifacts', exist_ok=True)
os.makedirs('/content/portfolio_release/reports', exist_ok=True)

with open('/content/portfolio_release/artifacts/id_mappings.pkl', 'wb') as f:
    pickle.dump({'visitor_to_idx': visitor_to_idx, 'idx_to_item': idx_to_item}, f)

!cp /content/two_tower_weights.pt /content/portfolio_release/artifacts/
!cp /content/faiss_item.index /content/portfolio_release/artifacts/
!cp /content/lgbm_ranker_tuned.txt /content/portfolio_release/artifacts/
!cp /content/scalers.pkl /content/portfolio_release/artifacts/
!cp /content/ltr_meta.pkl /content/portfolio_release/artifacts/
!cp /content/final_benchmark.csv /content/portfolio_release/reports/

with open('/content/portfolio_release/requirements.txt', 'w') as f:
    f.write("numpy\npandas\npyarrow\nscikit-learn\nscipy\ntorch\nfaiss-cpu\nlightgbm\noptuna\nshap\nmlflow\nplotly\n")

!cd /content && zip -rq /content/retailrocket_recsys_portfolio.zip portfolio_release
print("Paquete listo para descargar: /content/retailrocket_recsys_portfolio.zip")

Visitor ID: 2 | Latencia End-to-End: 113.56 ms
 rank  itemid  categoryid  tt_score  ltr_score
    1  216305         299  0.687580   2.222069
    2  325215         299  0.720591   2.133714
    3  259884         299  0.678293   1.561342
    4  342816         444  0.757040  -0.662942
    5   48072         299  0.709566  -4.566471
    6  140853         299  0.706687  -4.572051
    7   90389         684  0.703365  -4.581023
    8  159822         684  0.725596  -4.673198
    9  342264         444  0.743097  -4.727586
   10  154034         444  0.741295  -4.730100
Paquete listo para descargar: /content/retailrocket_recsys_portfolio.zip


In [14]:
from google.colab import files
files.download('/content/retailrocket_recsys_portfolio.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>